# Score Analysis: judge outputs, breakdowns, statistics

**Purpose:** Comprehensive analysis of LLM judge scores (FUN, NSI, INSI, ISI) across the conformity dataset. Loads scores with event metadata, computes quantile statistics, plots marginal histograms and category breakdowns, and identifies all-zero score patterns.

**Scope and segmentations:** All helpers support per-scope queries over `events`, `cleaned`, or `scores` tables. Score statistics are grouped by repo, author_association, event type, user, and combinations thereof. Statistics include count, mean, median, and quantiles (p25, p50, p75, p99).

**Reference:** Table structure in [`docs/DB_SCHEMA.md`](../docs/DB_SCHEMA.md). GitHub `author_association` enum values (OWNER, MEMBER, CONTRIBUTOR, FIRST_TIMER, NONE, etc.) and event type strings (`IssueCommentEvent`, `PullRequestEvent`, etc.) are documented there.

In [ ]:
import sys
from pathlib import Path

# Add repo root to sys.path so notebooks.lib can be imported
here = Path.cwd().resolve()
for p in [here, *here.parents]:
    if (p / "project_config.py").is_file():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        break

In [ ]:
import json
from html import escape

import pandas as pd
from IPython.display import HTML, Markdown, display

from notebooks.lib import (
    connect,
    DB_PATH,
    breakdown_repo_author_association,
    breakdown_event_type,
    breakdown_repo_event_type,
    load_scores_with_metadata,
    all_zero_scores_mask,
    score_stats_by_repo,
    score_stats_by_repo_author_association,
    score_stats_by_repo_event_type,
    score_stats_by_repo_aa_user,
    score_stats_by_repo_aa_user_event_type,
    total_score_by_repo_table,
    plot_global_score_histograms_all_vs_non_full_zero,
    plot_score_summaries_by_category,
    plot_total_score_by_repo,
    display_dataframe_scrollable,
    display_all_zero_score_comment_samples,
)

# Sample event ID for deep JSON inspection (None = first row in events)
SAMPLE_EVENT_ID: str | None = None

pd.set_option("display.max_rows", 20)
pd.set_option("display.max_colwidth", None)

print(f"DB: {DB_PATH.resolve()}  exists={DB_PATH.is_file()}")

## Available Scopes

All breakdown queries support segmentation by `events`, `cleaned`, or `scores` scope. Use the cells below to examine coverage and row counts.

In [ ]:
display(Markdown("### Scope: events (all raw GitHub events)"))
display(Markdown("**Repo × author_association**"))
display(breakdown_repo_author_association("events").head(10))
display(Markdown("**Event type — marginal**"))
display(breakdown_event_type("events"))

In [ ]:
display(Markdown("### Scope: cleaned (after preprocessing)"))
display(Markdown("**Repo × author_association**"))
display(breakdown_repo_author_association("cleaned").head(10))
display(Markdown("**Event type — marginal**"))
display(breakdown_event_type("cleaned"))

In [ ]:
display(Markdown("### Scope: scores (LLM judge rows)"))
display(Markdown("**Repo × author_association**"))
display(breakdown_repo_author_association("scores"))
display(Markdown("**Event type — marginal**"))
display(breakdown_event_type("scores"))

## Score Statistics and Distributions

Loads scores with event metadata (repo, author_association, user_login, event_type). Statistics include count, mean, median, and quantiles per group. Optional filtering for all-zero score rows.

### By Repo

In [ ]:
display(Markdown("**Score stats by repo**"))
display(score_stats_by_repo())

display(Markdown("**Total score (FUN+NSI+INSI+ISI) by repo** — per-row totals range 0–12"))
display(total_score_by_repo_table(load_scores_with_metadata()))

### By Repo × Author Association

In [ ]:
display(Markdown("**Score stats by repo × author_association**"))
display_dataframe_scrollable(score_stats_by_repo_author_association())

### By Repo × Event Type

In [ ]:
display(Markdown("**Score stats by repo × event type** — top-level `$.type` from `events.event_data`"))
pd.set_option("display.max_rows", 500)
display_dataframe_scrollable(score_stats_by_repo_event_type())
pd.set_option("display.max_rows", 20)

### By Repo × Author Association × User Login

In [ ]:
display(Markdown(
    "**Score stats by repo × author_association × user_login**\n\n"
    "User login: `payload.comment.user.login` / `review.user.login`, else `actor.login`"
))
pd.set_option("display.max_rows", 500)
display_dataframe_scrollable(score_stats_by_repo_aa_user())
pd.set_option("display.max_rows", 20)

### By Repo × Author Association × User Login × Event Type

In [ ]:
display(Markdown(
    "**Score stats by repo × author_association × user_login × event type** — 4-way breakdown"
))
pd.set_option("display.max_rows", 800)
display_dataframe_scrollable(
    score_stats_by_repo_aa_user_event_type(),
    max_height="36rem"
)
pd.set_option("display.max_rows", 20)

## Visualizations: Score Distributions

### Global Marginal Histograms

Counts at ordinal scores 0, 1, 2, 3 for each of FUN, NSI, INSI, ISI. Two figures: full cohort vs after removing all-zero rows.

In [ ]:
df_plot = load_scores_with_metadata()
for c in ["fun_score", "nsi_score", "insi_score", "isi_score"]:
    df_plot[c] = pd.to_numeric(df_plot[c], errors="coerce").fillna(0).astype(int)

plot_global_score_histograms_all_vs_non_full_zero(df_plot, compare_full_population=True)

### By Repo

Stacked counts at ordinal 0/1/2/3 per repo (full-zero judge rows excluded).

In [ ]:
plot_total_score_by_repo(df_plot)
plot_score_summaries_by_category(
    df_plot,
    category_col="repo",
    title="Repo-level judge scores"
)

### By Event Type

Stacked counts per event type (top-level `$.type`). Four separate figures per score dimension for independent saving. Horizontal bars with tight margins on y-axis. Full-zero judge rows excluded.

In [ ]:
plot_score_summaries_by_category(
    df_plot,
    category_col="event_type",
    title="Event-type-level judge scores",
    one_figure_per_panel=True,
)

In [ ]:
display(Markdown("**Same grouping, stacks omit s=0** (only s=1,2,3)"))
plot_score_summaries_by_category(
    df_plot,
    category_col="event_type",
    title="Event-type-level judge scores",
    include_s0=False,
    one_figure_per_panel=True,
    show_event_type_label_table=False,
)

## All-Zero Score Analysis

Rows where FUN=NSI=INSI=ISI=0 indicate the judge assigned no signal on any dimension. Show count and samples.

In [ ]:
df_full = load_scores_with_metadata(include_cleaned_text=True)
az = all_zero_scores_mask(df_full)
n_az = int(az.sum())

display(Markdown(
    f"#### All-zero scores (FUN=NSI=INSI=ISI=0)\n\n"
    f"**{n_az}** of **{len(df_full)}** loaded judge rows ({100*n_az/len(df_full):.1f}%)"
))

if n_az > 0:
    display(Markdown("#### Sample `cleaned_text` (truncated; scroll)"))
    display_all_zero_score_comment_samples(df_full, limit=500)